# CS F407 - Artificial Intelligence
## Laboratory: Search and A*

**Student:** Eshan Saikia  
**ID:** 2023A7PS1024G

This notebook contains the executable implementation and validation evidence for the Search laboratory. The accompanying report contains the written formulation, interpretation, and reflections.

## 1. Search Problem Specification

The warehouse supplied in the laboratory is:

```text
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
```

We formulate the search problem as \(P=(S,A,T,s_0,G,c)\).

| Component | Specification |
|---|---|
| State set \(S\) | Reachable grid positions `(row, column)` that are not obstacles |
| Actions \(A\) | Up, Down, Left, Right |
| Transition \(T\) | Move one cell in the chosen direction when the destination is valid |
| Initial state \(s_0\) | Position of `S` |
| Goal set \(G\) | Position of `G` |
| Cost \(c\) | 1 per movement |

A state is specified by the robot's current grid position. An action is invalid if it leaves the grid or enters an obstacle. The problem is deterministic because a valid state-action pair has a unique successor. A solution is a sequence of legal actions that reaches the goal.

In [1]:
from collections import deque
import heapq
import itertools
import math

WAREHOUSE = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
]

ACTIONS = [
    ("Up", (-1, 0)),
    ("Down", (1, 0)),
    ("Left", (0, -1)),
    ("Right", (0, 1)),
]

def find_symbol(grid, symbol):
    positions = [
        (r, c)
        for r, row in enumerate(grid)
        for c, value in enumerate(row)
        if value == symbol
    ]
    if len(positions) != 1:
        raise ValueError(f"Expected exactly one {symbol!r}, found {len(positions)}.")
    return positions[0]

START = find_symbol(WAREHOUSE, "S")
GOAL = find_symbol(WAREHOUSE, "G")

print("Start:", START)
print("Goal:", GOAL)

Start: (1, 1)
Goal: (7, 15)


## 2. Agent Design

The state is a coordinate pair `(row, column)`. The warehouse is represented as a list of equal-length strings. Valid actions are generated from the four possible displacements and checked against grid boundaries and obstacles.

A* uses

\[
f(n)=g(n)+h(n),
\]

where \(g(n)\) is the accumulated cost from the start and \(h(n)\) estimates the remaining cost.

For the main experiment:

\[
h(n)=|r-r_G|+|c-c_G|.
\]

The frontier is a priority queue ordered by \(f(n)\). The implementation retains the best known \(g\)-value for each discovered state and parent/action records for path reconstruction.

## 3. LLM-Assisted Implementation

### Exact A* implementation prompt

> I am implementing a simple warehouse search agent in Python. The environment is the supplied ASCII grid. `S` is the start, `G` is the goal, `#` is an obstacle, and `.` is free. The robot moves Up, Down, Left, or Right and every move costs 1. Implement A* search using Manhattan distance \(h(n)=|r-r_G|+|c-c_G|\). Represent positions as states, maintain a priority-queue frontier, track \(g(n)\), \(h(n)\), and \(f(n)=g(n)+h(n)\), avoid unnecessary repeated expansion, store parent/action information, reconstruct the path, and report solution found, path, path length, and states expanded. Keep it dependency-free and easy to inspect.

The A* search logic generated from this specification was accepted after inspection. I added the systematic test suite, explicit path-legality assertions, BFS comparison, and heuristic experiments. These additions do not change the search problem or A* formulation.

Before execution, the source was checked for the required state representation, successor generation, frontier, cost and heuristic calculations, goal test, duplicate handling, and path reconstruction.

In [2]:
def astar(grid, heuristic):
    """A* search with a priority queue and best-known path costs."""
    start = find_symbol(grid, "S")
    goal = find_symbol(grid, "G")

    counter = itertools.count()
    best_g = {start: 0}
    parent = {start: (None, None)}

    start_h = heuristic(start, goal)
    frontier = [(start_h, start_h, 0, next(counter), start)]

    # Best g-value at which a state has actually been expanded.
    expanded_at_g = {}
    states_expanded = 0

    while frontier:
        f, h, current_g, _, current = heapq.heappop(frontier)

        # Ignore stale queue entries superseded by a better path.
        if current_g != best_g.get(current):
            continue

        # Reopen only when a strictly better g-value has been found.
        if current in expanded_at_g and expanded_at_g[current] <= current_g:
            continue

        expanded_at_g[current] = current_g
        states_expanded += 1

        if current == goal:
            break

        for action_name, (dr, dc) in ACTIONS:
            nr, nc = current[0] + dr, current[1] + dc
            neighbour = (nr, nc)

            if (
                0 <= nr < len(grid)
                and 0 <= nc < len(grid[0])
                and grid[nr][nc] != "#"
            ):
                tentative_g = current_g + 1

                if tentative_g < best_g.get(neighbour, math.inf):
                    best_g[neighbour] = tentative_g
                    parent[neighbour] = (current, action_name)

                    nh = heuristic(neighbour, goal)
                    nf = tentative_g + nh
                    heapq.heappush(
                        frontier,
                        (nf, nh, tentative_g, next(counter), neighbour),
                    )

    if goal not in best_g:
        return {
            "found": False,
            "path": None,
            "actions": None,
            "length": None,
            "expanded": states_expanded,
        }

    path = []
    actions = []
    current = goal

    while current != start:
        previous, action = parent[current]
        path.append(current)
        actions.append(action)
        current = previous

    path.append(start)
    path.reverse()
    actions.reverse()

    return {
        "found": True,
        "path": path,
        "actions": actions,
        "length": len(actions),
        "expanded": states_expanded,
    }

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def euclidean(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def zero_heuristic(a, b):
    return 0

def twice_manhattan(a, b):
    return 2 * manhattan(a, b)

## 4. Task 2 - Main A* Run

In [3]:
main_astar = astar(WAREHOUSE, manhattan)

print("Solution found:", main_astar["found"])
print("Path:", main_astar["path"])
print("Actions:", main_astar["actions"])
print("Path length:", main_astar["length"])
print("States expanded:", main_astar["expanded"])

Solution found: True
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
Actions: ['Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Up', 'Up', 'Left', 'Left', 'Left', 'Left', 'Left', 'Left', 'Up', 'Up', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Right', 'Down', 'Down', 'Down', 'Down', 'Down', 'Down']
Path length: 40
States expanded: 64


In [4]:
# Independent validation of the A* result.
assert main_astar["found"]
assert main_astar["path"][0] == START
assert main_astar["path"][-1] == GOAL
assert main_astar["length"] == len(main_astar["path"]) - 1

action_delta = dict(ACTIONS)

for current, action, nxt in zip(
    main_astar["path"], main_astar["actions"], main_astar["path"][1:]
):
    delta = (nxt[0] - current[0], nxt[1] - current[1])
    assert delta == action_delta[action]
    assert WAREHOUSE[nxt[0]][nxt[1]] != "#"

print("Main A* path validation passed.")

Main A* path validation passed.


In [5]:
def render_path(grid, path):
    path_set = set(path or [])
    rendered = []

    for r, row in enumerate(grid):
        chars = []
        for c, cell in enumerate(row):
            if (r, c) in path_set and cell not in {"S", "G"}:
                chars.append("*")
            else:
                chars.append(cell)
        rendered.append("".join(chars))

    return rendered

print("Warehouse with A* path marked by '*':")
for row in render_path(WAREHOUSE, main_astar["path"]):
    print(row)

Warehouse with A* path marked by '*':
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################


## 5. Task 3 - Systematic Tests

The four required tests are the original warehouse, a one-step case, an inaccessible-goal case, and a map with multiple possible routes.

In [6]:
TRIVIAL = [
    "#####",
    "#SG##",
    "#####",
]

NO_SOLUTION = [
    "#######",
    "#S....#",
    "###.###",
    "#...#G#",
    "#######",
]

ALTERNATIVE = [
    "#######",
    "#S...G#",
    "#.#.#.#",
    "#.....#",
    "#######",
]

def run_test(name, grid, heuristic=manhattan):
    result = astar(grid, heuristic)
    print(f"\n{name}")
    print("Found:", result["found"])
    print("Path:", result["path"])
    print("Actions:", result["actions"])
    print("Length:", result["length"])
    print("States expanded:", result["expanded"])
    return result

trivial_result = run_test("Test 2 - Trivial case", TRIVIAL)
unsolved_result = run_test("Test 3 - No solution", NO_SOLUTION)
alternative_result = run_test("Test 4 - Alternative paths", ALTERNATIVE)

assert trivial_result["found"] and trivial_result["length"] == 1
assert not unsolved_result["found"]
assert alternative_result["found"]


Test 2 - Trivial case
Found: True
Path: [(1, 1), (1, 2)]
Actions: ['Right']
Length: 1
States expanded: 2

Test 3 - No solution
Found: False
Path: None
Actions: None
Length: None
States expanded: 9

Test 4 - Alternative paths
Found: True
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]
Actions: ['Right', 'Right', 'Right', 'Right']
Length: 4
States expanded: 5


In [7]:
def bfs(grid):
    start = find_symbol(grid, "S")
    goal = find_symbol(grid, "G")

    frontier = deque([start])
    parent = {start: (None, None)}
    states_expanded = 0

    while frontier:
        current = frontier.popleft()
        states_expanded += 1

        if current == goal:
            break

        for action_name, (dr, dc) in ACTIONS:
            nr, nc = current[0] + dr, current[1] + dc
            neighbour = (nr, nc)

            if (
                0 <= nr < len(grid)
                and 0 <= nc < len(grid[0])
                and grid[nr][nc] != "#"
                and neighbour not in parent
            ):
                parent[neighbour] = (current, action_name)
                frontier.append(neighbour)

    if goal not in parent:
        return {
            "found": False,
            "path": None,
            "actions": None,
            "length": None,
            "expanded": states_expanded,
        }

    path, actions = [], []
    current = goal

    while current != start:
        previous, action = parent[current]
        path.append(current)
        actions.append(action)
        current = previous

    path.append(start)
    path.reverse()
    actions.reverse()

    return {
        "found": True,
        "path": path,
        "actions": actions,
        "length": len(actions),
        "expanded": states_expanded,
    }

alternative_bfs = bfs(ALTERNATIVE)

print("Alternative-path A* length:", alternative_result["length"])
print("Alternative-path BFS length:", alternative_bfs["length"])
print("A* found a shortest path:", alternative_result["length"] == alternative_bfs["length"])

assert alternative_bfs["found"]
assert alternative_result["length"] == alternative_bfs["length"]

Alternative-path A* length: 4
Alternative-path BFS length: 4
A* found a shortest path: True


## 6. Task 4 - Inspecting the A* Algorithm

| Concept | Implementation |
|---|---|
| State | `(row, column)` tuple |
| Action | `ACTIONS` |
| Transition | Add direction displacement |
| Goal test | `current == goal` |
| \(g(n)\) | `current_g` / `best_g` |
| \(h(n)\) | `heuristic(...)` |
| \(f(n)\) | `tentative_g + nh` |
| Frontier | `heapq` priority queue |
| Visited/discovered records | `best_g`, `expanded_at_g` |
| Path reconstruction | `parent` and stored action |

The priority queue selects the smallest \(f(n)\). Stale entries are ignored when a better \(g\)-value has already been found.

## 7. Task 5 - BFS vs A*

The same original warehouse is used for both methods. Under the notebook's convention, a state is counted when it is actually expanded, including the goal. Both algorithms find a length-40 path and both expand 64 states on this map and action order.

Therefore, **neither algorithm expands fewer states in this particular experiment**. A* can reduce unnecessary exploration when the heuristic provides a stronger distinction between frontier states.

In [8]:
main_bfs = bfs(WAREHOUSE)

print(f"{'Algorithm':<10} {'Found':<8} {'Length':<8} {'Expanded':<10}")
print("-" * 42)
for name, result in [("BFS", main_bfs), ("A*", main_astar)]:
    print(
        f"{name:<10} {str(result['found']):<8} "
        f"{str(result['length']):<8} {result['expanded']:<10}"
    )

assert main_bfs["found"] and main_astar["found"]
assert main_bfs["length"] == main_astar["length"] == 40
assert main_bfs["expanded"] == main_astar["expanded"] == 64

Algorithm  Found    Length   Expanded  
------------------------------------------
BFS        True     40       64        
A*         True     40       64        


## 8. Task 6 - Heuristic Investigation

The required heuristic variants are:

\[
h(n)=0,\qquad
h_M(n)=|r-r_G|+|c-c_G|,
\]

\[
h_E(n)=\sqrt{(r-r_G)^2+(c-c_G)^2},
\qquad
2h_M(n).
\]

For this four-neighbour unit-cost grid, Manhattan distance is admissible because each coordinate difference requires at least that many moves. Euclidean distance is also admissible because it is no larger than Manhattan distance. The zero heuristic is admissible but provides no goal-directed information. Multiplying Manhattan distance by 2 can overestimate the remaining cost and is therefore not admissible in general.

In [9]:
heuristics = [
    ("h(n) = 0", zero_heuristic),
    ("Manhattan", manhattan),
    ("Euclidean", euclidean),
    ("2 x Manhattan", twice_manhattan),
]

heuristic_results = []
for name, heuristic in heuristics:
    result = astar(WAREHOUSE, heuristic)
    heuristic_results.append((name, result["found"], result["length"], result["expanded"]))

print(f"{'Heuristic':<18} {'Found':<8} {'Length':<8} {'Expanded':<10}")
print("-" * 50)
for name, found, length, expanded in heuristic_results:
    print(f"{name:<18} {str(found):<8} {str(length):<8} {expanded:<10}")

Heuristic          Found    Length   Expanded  
--------------------------------------------------
h(n) = 0           True     40       64        
Manhattan          True     40       64        
Euclidean          True     40       64        
2 x Manhattan      True     40       68        


### Heuristic interpretation

- **\(h=0\):** A* becomes uniform-cost search; with unit edge costs this has the same expansion behaviour as BFS here.
- **Manhattan:** natural for horizontal/vertical movement and admissible for this problem.
- **Euclidean:** also admissible, but it produced the same expansion count as Manhattan on this particular map.
- **\(2h_M\):** non-admissible in general. It still found the length-40 path here, but expanded 68 states rather than 64, so this instance does not demonstrate a shortest-path guarantee for the scaled heuristic.

The experiment shows that heuristic quality affects search order. The effect is instance-dependent.


## 9. Task 7 - Evaluation of the LLM-Generated Agent

1. **What parts of the generated code were correct immediately?** The state representation, four-direction successor generation, Manhattan heuristic, \(g/h/f\) calculations, priority-queue frontier, goal test, and parent-based reconstruction matched the specification.

2. **Did you find any bugs or design problems?** No functional bug was found in the first execution. Additional validation was added to test the implementation rather than assume correctness.

3. **How did you discover those problems?** The implementation was inspected against the formal search specification and then exercised with known-answer tests: a one-step solution, an inaccessible goal, and a map with alternative paths.

4. **Did the LLM use terminology or data structures that you did not understand?** No. The implementation used standard A* concepts such as a priority queue, \(g(n)\), \(h(n)\), \(f(n)\), and parent records, which were inspected directly.

5. **Did you modify the LLM-generated code?** Yes. I added the systematic test harness, path-validity assertions, BFS comparison, heuristic variants, and explicit stale-frontier-entry handling. These changes support validation and robustness without changing the search formulation.

6. **Which tests were most useful?** The no-solution test verified termination and failure reporting, while the alternative-path test verified shortest-path behaviour against an independently computed BFS result.

7. **Could you have trusted the program without testing it?** No. A program can run and still implement incorrect transitions, priorities, goal handling, or path reconstruction.

8. **What did you understand about A* that you did not understand before implementing it?** Implementing the frontier made the role of \(f=g+h\) concrete: the heuristic does not replace the path cost already incurred; it changes which frontier state is explored next.


## 10. Final Reflection

### 1. Why formulate the search problem before writing the search algorithm?

Formulating the state, actions, transitions, initial state, goal, and cost defines the problem independently of the implementation. It provides a specification against which an A* implementation can be checked.

### 2. In what sense is A* an informed search algorithm?

A* uses an estimate \(h(n)\) of the remaining cost to the goal in addition to the accumulated cost \(g(n)\). Its priority \(f(n)=g(n)+h(n)\) therefore uses problem-specific information to choose the next state.

### 3. Why does the choice of heuristic matter?

The heuristic controls the order in which frontier states are explored. An admissible and informative heuristic can guide the search toward the goal while preserving shortest-path guarantees; an overly aggressive heuristic can lose admissibility.

### 4. What did the LLM contribute to the engineering process?

The LLM translated the specified A* design into executable Python and provided a starting structure for the frontier, cost calculations, and path reconstruction. The implementation was then inspected and tested independently.

### 5. What could go wrong if generated code were accepted without testing?

The program could produce plausible output while mishandling obstacles, state transitions, duplicate states, priorities, or path reconstruction. Known-answer tests are necessary to establish that the implementation actually satisfies the search specification.


## 11. Verification Checklist

- [x] Formal search problem \(P=(S,A,T,s_0,G,c)\).
- [x] Agent design and A* components.
- [x] LLM implementation prompt and implementation review.
- [x] Original-warehouse A* run.
- [x] Trivial, no-solution, and alternative-path tests.
- [x] A* code inspection mapping.
- [x] BFS vs A* comparison.
- [x] Zero, Manhattan, Euclidean, and \(2\times\) Manhattan heuristic experiments.
- [x] Task 7 questions and final reflection.
